In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# -------------------------------------------------------------
# 0. Global Dark Configuration
# -------------------------------------------------------------
plt.style.use('dark_background')

# -------------------------------------------------------------
# 1. Physics Parameters: 2 Full Cycloid Periods
# -------------------------------------------------------------
R = 1.0                             # Cyclotron radius (Larmor radius)
omega = 1.5                         # Cyclotron frequency (rad/s)
v0 = omega * R                      # Drift velocity parameter
T_one_loop = 2.0 * np.pi / omega    # Period for one cusp loop (~4.19 s)
T_total = 2.0 * T_one_loop          # 2 Full loops to show recurrence (~8.38 s)

n_frames = 160
t = np.linspace(0, T_total, n_frames)

# Frame S' (Moving Box) Cycloidal Trajectory
x_prime = R * (np.sin(omega * t) - omega * t)
y_prime = R * (1.0 - np.cos(omega * t))

# Velocity components in S'
vx_prime = v0 * (np.cos(omega * t) - 1.0)
vy_prime = v0 * np.sin(omega * t)

# Scaled Force Vectors for visualization
# F_elec is constant along +y'
scale_F = 0.75
f_ex = np.zeros_like(t)
f_ey = np.full_like(t, scale_F)

# F_mag = q(v' x B') = -q vy B0 x' + q vx B0 y'
f_bx = -scale_F * np.sin(omega * t)
f_by = scale_F * (np.cos(omega * t) - 1.0)

# F_net = F_elec + F_mag
f_nx = f_ex + f_bx
f_ny = f_ey + f_by

# -------------------------------------------------------------
# 2. Figure Setup (Frame S' Perspective)
# -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 5.0), dpi=100, facecolor='#000000')
ax.set_facecolor('#000000')

ax.set_xlim(-13.5, 1.2)
ax.set_ylim(-0.4, 2.7)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', color='#1e293b', alpha=0.6)

for spine in ax.spines.values():
    spine.set_color('#334155')
ax.tick_params(colors='#94a3b8', labelsize=8.5)

ax.set_title("Frame S' (Inside Compartment): The Dual-Field Cycloid & Force Dynamics",
             fontsize=11, pad=14, color='#ffffff', weight='bold')
ax.set_xlabel("x' (Displacement relative to compartment origin)", fontsize=9, color='#94a3b8')
ax.set_ylabel("y' (Height above floor)", fontsize=9, color='#94a3b8')

# -------------------------------------------------------------
# 3. Static Environment: Fields, Floor & Cusps
# -------------------------------------------------------------
# Floor Line
ax.axhline(0.0, color='#475569', lw=1.8, linestyle='-', alpha=0.9)
ax.text(0.9, -0.22, "Floor", color='#64748b', fontsize=8, ha='right')

# Background Magnetic Field (into page)
gx, gy = np.meshgrid(np.linspace(-13.0, 0.5, 15), np.linspace(0.3, 2.3, 4))
ax.scatter(gx, gy, marker='o', s=60, facecolors='none', edgecolors='#38bdf8', alpha=0.20, lw=1.0)
ax.scatter(gx, gy, marker='x', s=25, color='#38bdf8', alpha=0.20, lw=1.0)

# Background Electric Field (Uniform upward arrows)
for ex in np.linspace(-12.5, 0.2, 12):
    ax.annotate('', xy=(ex, 2.3), xytext=(ex, 0.15),
                arrowprops=dict(arrowstyle="->", color='#facc15', lw=1.1, alpha=0.25))

# Reference Cusp Markers
cusp_x = [0.0, -2.0 * np.pi * R, -4.0 * np.pi * R]
cusp_labels = ["Cusp 1 (t'=0)", r"Cusp 2 (t'=T)", r"Cusp 3 (t'=2T)"]
for cx, clabel in zip(cusp_x, cusp_labels):
    ax.axvline(cx, color='#334155', linestyle=':', lw=1.2, alpha=0.7)
    ax.text(cx, -0.32, clabel, color='#94a3b8', fontsize=7.5, ha='center')

# Ceiling dashed boundary
ax.axhline(2.0 * R, color='#38bdf8', linestyle='--', lw=0.9, alpha=0.35)
ax.text(-13.2, 2.05, r"Peak Altitude ($y'_{\max} = 2R$)", color='#38bdf8', fontsize=7.5, alpha=0.7)

# -------------------------------------------------------------
# 4. Dynamic Animated Artists
# -------------------------------------------------------------
# Cycloid Trail & Particle
trail, = ax.plot([], [], color='#f43f5e', lw=2.2, alpha=0.85)
particle, = ax.plot([], [], 'o', color='#ff3366', markersize=8.5,
                    markeredgecolor='#ffffff', markeredgewidth=1.1)

# Dynamic Force Vector Lines & Tips
line_Fe, = ax.plot([], [], color='#facc15', lw=2.0, alpha=0.9)
tip_Fe, = ax.plot([], [], '^', color='#facc15', markersize=4.5)

line_Fb, = ax.plot([], [], color='#38bdf8', lw=2.0, alpha=0.9)
tip_Fb, = ax.plot([], [], 'o', color='#38bdf8', markersize=4.0)

line_Fn, = ax.plot([], [], color='#c084fc', lw=2.0, linestyle='--', alpha=0.9)
tip_Fn, = ax.plot([], [], 's', color='#c084fc', markersize=4.0)

# Real-time HUD Status & Force Legend
hud_phase = ax.text(-13.0, 2.52, "", fontsize=8.5, color='#ffffff', weight='bold')

# Fixed Legend Box at Top-Right
ax.plot([-3.8, -3.3], [2.52, 2.52], color='#facc15', lw=2)
ax.text(-3.1, 2.48, r"$\mathbf{F}'_{\mathrm{elec}}$", color='#facc15', fontsize=8)

ax.plot([-2.0, -1.5], [2.52, 2.52], color='#38bdf8', lw=2)
ax.text(-1.3, 2.48, r"$\mathbf{F}'_{\mathrm{mag}}$", color='#38bdf8', fontsize=8)

ax.plot([-0.2, 0.3], [2.52, 2.52], color='#c084fc', lw=2, linestyle='--')
ax.text(0.5, 2.48, r"$\mathbf{F}'_{\mathrm{net}}$", color='#c084fc', fontsize=8)

# -------------------------------------------------------------
# 5. Animation Update Function
# -------------------------------------------------------------
def update(i):
    px, py = x_prime[i], y_prime[i]
    
    # Update trail and particle position
    trail.set_data(x_prime[:i+1], y_prime[:i+1])
    particle.set_data([px], [py])
    
    # 1. Electric Force Vector (Always pointing up +y')
    line_Fe.set_data([px, px + f_ex[i]], [py, py + f_ey[i]])
    tip_Fe.set_data([px + f_ex[i]], [py + f_ey[i]])
    
    # 2. Magnetic Force Vector (Dynamic)
    line_Fb.set_data([px, px + f_bx[i]], [py, py + f_by[i]])
    tip_Fb.set_data([px + f_bx[i]], [py + f_by[i]])
    
    # 3. Net Force Vector (Resultant acceleration)
    line_Fn.set_data([px, px + f_nx[i]], [py, py + f_ny[i]])
    tip_Fn.set_data([px + f_nx[i]], [py + f_ny[i]])
    
    # Update Status Text depending on phase angle (mod 2pi)
    phase = (omega * t[i]) % (2.0 * np.pi)
    if phase < 0.25 or phase > (2.0 * np.pi - 0.25):
        hud_phase.set_text("Status: Cusp / Lift-off  [ F'_mag ≈ 0, F'_elec drives lift ]")
    elif phase < (np.pi - 0.25):
        hud_phase.set_text("Status: Upward Ascent  [ F'_mag actively pulls backward ]")
    elif phase <= (np.pi + 0.25):
        hud_phase.set_text("Status: Crest Peak (y'=2R) [ vy'=0, F'_net points downward ]")
    else:
        hud_phase.set_text("Status: Controlled Descent [ Steering smoothly toward next cusp ]")
    
    return trail, particle, line_Fe, tip_Fe, line_Fb, tip_Fb, line_Fn, tip_Fn, hud_phase

# -------------------------------------------------------------
# 6. Render, Save & Display in Kaggle
# -------------------------------------------------------------
anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)

# Export clean video with exact black borders
anim.save('dual_field_cycloid.mp4', writer='ffmpeg', fps=20,
          savefig_kwargs={'facecolor': '#000000'},
          extra_args=['-vcodec', 'libx264'])
plt.close(fig)

# In-Notebook HTML5 Player
HTML(anim.to_html5_video())